In [0]:
# Load the data from Silver table based on latest OrderDate (incremental load)
if spark.catalog.tableExists('practice.sales_gold.fact_sales'):
    last_load_date = spark.sql("""
        SELECT MAX(OrderDate) 
        FROM practice.sales_gold.fact_sales
    """).collect()[0][0]
    print('last_load_date:', last_load_date)
else:
    last_load_date = '1000-01-01'
    print('last_load_date:', last_load_date)

In [0]:
# Create Product dimension table (source view)
spark.sql(f"""
    WITH cte AS (
        SELECT DISTINCT 
            ProductId, 
            Category, 
            SubCategory, 
            'Y' AS is_current,
            CURRENT_TIMESTAMP() AS processed_date,
            DATE(CURRENT_TIMESTAMP()) AS start_date,
            DATE('3000-01-01') AS end_date
        FROM practice.sales_silver.s_sales 
        WHERE OrderDate > '{last_load_date}'
    )
    SELECT 
                * 
    FROM cte
""").createOrReplaceTempView('src')

In [0]:
%sql
-- View sample records from source view
SELECT * 
FROM src 
LIMIT 3;

In [0]:
# Create the dim_product table (actual target table)
spark.sql("""
    CREATE TABLE IF NOT EXISTS practice.sales_gold.dim_product (
        Product_SK BIGINT GENERATED ALWAYS AS IDENTITY,
        ProductId STRING,
        Category STRING,
        SubCategory STRING,
        is_current STRING,
        processed_date TIMESTAMP,
        start_date DATE,
        end_date DATE
    )
""")

In [0]:
# SCD Type 2 - UPDATE + MERGE data in dim_product
spark.sql("""
    MERGE INTO practice.sales_gold.dim_product AS tgr
    USING src
    ON src.ProductId = tgr.ProductId
        AND tgr.is_current = 'Y'
    
    WHEN MATCHED AND (
        src.Category <> tgr.Category
        OR src.SubCategory <> tgr.SubCategory
        
    )
    THEN UPDATE SET
        tgr.end_date = DATE(CURRENT_TIMESTAMP()),
        tgr.is_current = 'N'

 """)

In [0]:
# SCD Type 2 - UPDATE + MERGE data in dim_product
spark.sql("""
    MERGE INTO practice.sales_gold.dim_product AS tgr
    USING src
    ON src.ProductId = tgr.ProductId
        AND tgr.is_current = 'Y'
    
    WHEN NOT MATCHED 
    THEN INSERT (
        ProductId, 
        Category, 
        SubCategory, 
        is_current, 
        processed_date, 
        start_date, 
        end_date
    )
    VALUES (
        src.ProductId, 
        src.Category, 
        src.SubCategory, 
        src.is_current, 
        src.processed_date, 
        src.start_date, 
        src.end_date
    )
        
    """)

In [0]:
%sql
-- View sample records from dim_product
SELECT * 
FROM practice.sales_gold.dim_product 
ORDER BY Product_SK DESC 
LIMIT 5;

In [0]:
%sql
-- QC: Check specific products in dim_product
SELECT * 
FROM practice.sales_gold.dim_product 
WHERE ProductId IN ('OFF-PA-10004041', 'FUR-TA-10001039', 'OFF-PROD-DEMO1')
ORDER BY Product_SK 
LIMIT 10;

## Fact Table

In [0]:
last_load_date

In [0]:
# Create fact table source view with dimension keys
spark.sql(f"""
    SELECT 
        s.OrderId, 
        s.OrderDate, 
        s.PostalCode, 
        s.ShipMode,
        s.Segment,
        s.Country,
        s.City,
        s.State,
        s.Region,
        s.CostPrice, 
        s.ListPrice, 
        s.Quantity,
        s.DiscountPercent, 
        s.Total_Sales, 
        s.Final_Price,
        p.Product_SK

    FROM practice.sales_silver.s_sales s
    LEFT JOIN practice.sales_gold.dim_product p
        ON s.ProductId = p.ProductId 
        AND p.is_current = 'Y' 
    WHERE s.OrderDate > '{last_load_date}'
      
""").createOrReplaceTempView('src_fact')

In [0]:
%sql
-- View sample records from fact source view
SELECT * 
FROM src_fact 
ORDER BY OrderId DESC 
LIMIT 5;

In [0]:
# Create fact_sales table
spark.sql("""
    CREATE TABLE IF NOT EXISTS practice.sales_gold.fact_sales (
        OrderId STRING,
        OrderDate DATE,
        PostalCode STRING,
        ShipMode STRING,
        Segment STRING,
        Country STRING,
        City STRING,
        State STRING,
        Region STRING,
        CostPrice DOUBLE,
        ListPrice DOUBLE,
        Quantity INT,
        DiscountPercent DOUBLE,
        Total_Sales DOUBLE,
        Final_Price DOUBLE,
        Product_SK INT
    )
""")

In [0]:
# Insert data from source view into fact_sales table
spark.sql("""
    INSERT INTO practice.sales_gold.fact_sales
    SELECT * 
    FROM src_fact
""")

In [0]:
%sql
-- Count records in fact_sales
SELECT 
    COUNT(*) AS total_records, 
    COUNT(DISTINCT OrderId) AS distinct_orders, 
    COUNT(OrderId) AS non_null_orders
FROM practice.sales_gold.fact_sales;

In [0]:
%sql
-- View sample records from fact_sales
SELECT * 
FROM practice.sales_gold.fact_sales 
ORDER BY OrderId DESC 
LIMIT 5;

In [0]:
%sql
-- QC: Check for duplicate OrderIds
SELECT 
    OrderId, 
    COUNT(*) AS order_count
FROM practice.sales_gold.fact_sales 
GROUP BY OrderId
HAVING COUNT(*) > 1;

## QC

In [0]:
%sql
select p.Category, round(sum(s.Final_Price),0) as Final
from practice.sales_gold.fact_sales s
inner join practice.sales_gold.dim_product p 
on s.Product_SK = p.Product_SK
where p.is_current = 'Y'
group by p.Category

In [0]:
%sql
select * from practice.sales_gold.dim_product
where Category = 'Demo Products' or SubCategory = 'Demo Paper'

In [0]:
%sql
select * from practice.sales_gold.fact_sales where Product_SK = '1864'

In [0]:
%sql
select p.Category, round(sum(s.ListPrice),0) as Final
from practice.sales_gold.fact_sales s
inner join practice.sales_gold.dim_product p 
on s.Product_SK = p.Product_SK
where p.is_current = 'Y'
group by p.Category order by 1

In [0]:
%sql
select *
from practice.sales_gold.fact_sales s
inner join practice.sales_gold.dim_product p 
on s.Product_SK = p.Product_SK
order by s.OrderId
--where p.is_current = 'Y'

In [0]:
%sql
select s.Category, round(sum(s.ListPrice),0) as Final
from practice.sales_silver.s_sales s
group by s.Category
order by 1